# Fungi-Only Effector Inference (Google Colab)

This notebook predicts **Effector** vs **Non-Effector** labels for fungal protein sequences using a pretrained PEACE prototype model.

## What this notebook does
1. Clone this repository and install dependencies.
2. Load the pretrained model from `pretrained_models/fungus_model`.
3. Accept user input as either:
   - one raw amino-acid sequence, or
   - an uploaded FASTA file.
4. Validate input with safety limits to prevent out-of-memory failures.
5. Generate ProtT5 embeddings (8 SimCSE-style variants).
6. Run fold-ensemble prototype inference.
7. Show results in a table and export CSV.

## Recommendation
Use **Runtime -> Change runtime type -> GPU** for faster embedding extraction.


In [ ]:
#@title 1) Setup: clone repo and install dependencies
import os
import subprocess
import sys
from pathlib import Path

REPO_OWNER = "Structurebiology-BNL"
REPO_NAME = "effector_prediction_with_contrastive_learning"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
REPO_DIR = Path(f"/content/{REPO_NAME}")


if not REPO_DIR.exists():
    print(f"Cloning repository into {REPO_DIR}...")
    clone_result = subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        text=True,
        capture_output=True,
    )
    if clone_result.returncode != 0:
        raise RuntimeError(
            "Failed to clone the public repository. "
            f"\n\nGit output:\n{clone_result.stderr.strip()}"
        )
else:
    print(f"Reusing existing repository at: {REPO_DIR}")

os.chdir(REPO_DIR)

# Install package dependencies and SentencePiece tokenizer backend for ProtT5.
install_cmd = [sys.executable, "-m", "pip", "install", "-e", ".", "sentencepiece"]
install_result = subprocess.run(install_cmd, text=True, capture_output=True)
if install_result.returncode != 0:
    raise RuntimeError(
        "Dependency installation failed.\n\n"
        f"stdout:\n{install_result.stdout.strip()}\n\n"
        f"stderr:\n{install_result.stderr.strip()}"
    )

try:
    import effector_bincls  # noqa: F401
except ModuleNotFoundError:
    # Colab can occasionally lose editable-install path metadata; fallback to src.
    src_path = REPO_DIR / "src"
    if src_path.exists() and str(src_path) not in sys.path:
        sys.path.insert(0, str(src_path))
    try:
        import effector_bincls  # noqa: F401
    except ModuleNotFoundError as exc:
        raise RuntimeError(
            "Package install appeared to succeed but 'effector_bincls' is "
            "still not importable. Restart runtime and rerun this cell."
        ) from exc

print("Setup complete.")
print(f"Working directory: {Path.cwd()}")


In [ ]:
#@title 2) Import notebook modules
import io
import logging
import re
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from Bio import SeqIO
from IPython.display import display

from effector_bincls.data import open_packed_embedding_dataset
from effector_bincls.inference.prototype import run_inference
from effector_bincls.notebook_support import (
    extract_prott5_embeddings,
    load_notebook_model_metadata,
)
from effector_bincls.run_utils import load_run_config, resolve_device

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

WORK_DIR = Path("work/colab_fungus_inference")
EMBEDDING_ROOT_DIR = WORK_DIR / "embeddings"
OUTPUT_DIR = WORK_DIR / "outputs"
EMBEDDING_ROOT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Imports complete.")
print(f"Embedding output root: {EMBEDDING_ROOT_DIR}")
print(f"Results output directory: {OUTPUT_DIR}")


In [ ]:
#@title 3) Load pretrained model metadata and defaults
MODEL_DIR = Path("pretrained_models/fungus_model")

if not MODEL_DIR.exists():
    raise FileNotFoundError(
        "Could not find pretrained model folder at 'pretrained_models/fungus_model'. "
        "Please confirm the model files were copied into the repository."
    )

config_path = MODEL_DIR / "config.yml"
if not config_path.exists():
    raise FileNotFoundError(f"Missing required model config file: {config_path}")

MODEL_CONFIG = load_run_config(MODEL_DIR)
MODEL_METADATA = load_notebook_model_metadata(MODEL_DIR)
NOTEBOOK_METADATA = MODEL_METADATA["notebook"]

fold_one_two_stage = MODEL_DIR / "fold_1" / "finetuning" / "checkpoint.pt"
fold_one_single_stage = MODEL_DIR / "fold_1" / "checkpoint.pt"

if fold_one_two_stage.exists():
    IS_SINGLE_STAGE = False
    CHECKPOINT_LAYOUT = "two-stage (fold_X/finetuning/checkpoint.pt)"
elif fold_one_single_stage.exists():
    IS_SINGLE_STAGE = True
    CHECKPOINT_LAYOUT = "single-stage (fold_X/checkpoint.pt)"
else:
    raise FileNotFoundError(
        "Could not detect checkpoint layout. Expected either "
        "'fold_1/finetuning/checkpoint.pt' (two-stage) or "
        "'fold_1/checkpoint.pt' (single-stage)."
    )

NUM_FOLDS = int(getattr(MODEL_CONFIG.training, "num_folds", 0))
if NUM_FOLDS <= 0:
    fold_dirs = sorted(MODEL_DIR.glob("fold_*"))
    NUM_FOLDS = len(fold_dirs)
if NUM_FOLDS <= 0:
    raise ValueError(
        "Could not determine the number of folds from config or checkpoint "
        "directories."
    )

POOLING_TYPE = str(getattr(MODEL_CONFIG.features, "pooling_type", "mean"))
NORMALIZE_EMBEDDINGS = bool(getattr(MODEL_CONFIG.features, "normalize", True))
USE_VARIANTS = bool(getattr(MODEL_CONFIG.training, "use_variants", False))

print("Model defaults loaded:")
print(f"- Model directory: {MODEL_DIR}")
print(f"- Checkpoint layout: {CHECKPOINT_LAYOUT}")
print(f"- Number of folds: {NUM_FOLDS}")
print(f"- Pooling type: {POOLING_TYPE}")
print(f"- Normalize embeddings: {NORMALIZE_EMBEDDINGS}")
print(f"- Expects variants: {USE_VARIANTS}")


In [ ]:
#@title 4) Set default classification threshold
NOTEBOOK_DEFAULT_THRESHOLD = float(NOTEBOOK_METADATA["default_threshold"])
NOTEBOOK_THRESHOLD_METHOD = str(
    NOTEBOOK_METADATA.get("threshold_method", "configured")
)
NOTEBOOK_THRESHOLD_SOURCE = str(
    NOTEBOOK_METADATA.get("threshold_source", "metadata")
)

AUTO_THRESHOLD = NOTEBOOK_DEFAULT_THRESHOLD
print(f"Default threshold for this notebook run: {AUTO_THRESHOLD:.6f}")
print(
    "This default comes from "
    f"{MODEL_DIR / 'metadata.yml'} "
    f"({NOTEBOOK_THRESHOLD_METHOD}, {NOTEBOOK_THRESHOLD_SOURCE})."
)


## Input Options and Safety Limits

You can run this notebook in two ways:

- **Raw sequence**: paste a single amino-acid sequence.
- **Upload FASTA**: upload one FASTA file containing one or more sequences.

To reduce out-of-memory risk and keep runtime reasonable, the notebook enforces:

- maximum number of sequences per run, and
- maximum sequence length.

If limits are exceeded, the notebook raises a clear error message before embedding starts.


In [ ]:
#@title 5) User input form
INPUT_MODE = "Raw sequence" #@param ["Raw sequence", "Upload FASTA"]
RAW_SEQUENCE = "MKTAYIAKQRQISFVKSHFSRQDILDLIC" #@param {type:"string"}
MAX_SEQUENCES_ALLOWED = 20 #@param {type:"integer"}
MAX_SEQUENCE_LENGTH_ALLOWED = 2500 #@param {type:"integer"}
THRESHOLD_OVERRIDE = "" #@param {type:"string"}

print("Input form captured.")
print(f"- Input mode: {INPUT_MODE}")
print(f"- Max sequences allowed: {MAX_SEQUENCES_ALLOWED}")
print(f"- Max sequence length allowed: {MAX_SEQUENCE_LENGTH_ALLOWED}")
if THRESHOLD_OVERRIDE.strip():
    print(f"- Threshold override requested: {THRESHOLD_OVERRIDE}")
else:
    print(
        "- Threshold override: not set "
        f"(will use default threshold: {AUTO_THRESHOLD:.6f})"
    )


In [ ]:
#@title 6) Parse and validate input sequences
VALID_AMINO_ACIDS = set("ACDEFGHIKLMNPQRSTVWYX")


def sanitize_sequence(raw_sequence: str) -> str:
    """Normalize sequence text and map ambiguous residues to X."""
    cleaned = re.sub(r"\s+", "", raw_sequence).upper().replace("*", "")
    cleaned = re.sub(r"[UZOB]", "X", cleaned)
    return cleaned


def make_safe_id(original_id: str, used_ids: set[str], fallback_index: int) -> str:
    """Create a filesystem-safe unique sequence id for packed dataset rows."""
    candidate = re.sub(r"[^A-Za-z0-9_.-]", "_", original_id.strip())
    if not candidate:
        candidate = f"sequence_{fallback_index}"

    base = candidate
    suffix = 1
    while candidate in used_ids:
        suffix += 1
        candidate = f"{base}_{suffix}"

    used_ids.add(candidate)
    return candidate


def parse_sequences_from_raw(raw_sequence: str) -> list[dict]:
    """Parse one raw sequence from the form text box."""
    if not raw_sequence or not raw_sequence.strip():
        raise ValueError(
            "Raw sequence input is empty. Please paste a protein sequence in "
            "the RAW_SEQUENCE field."
        )

    return [
        {"sequence_id": "user_sequence_1", "sequence": sanitize_sequence(raw_sequence)}
    ]


def parse_sequences_from_uploaded_fasta() -> list[dict]:
    """Upload and parse a FASTA file in Google Colab."""
    if colab_files is None:
        raise EnvironmentError(
            "FASTA upload requires Google Colab "
            "(google.colab.files.upload)."
        )

    print("Please choose exactly one FASTA file (.fasta, .fa, or .fna).")
    uploaded = colab_files.upload()

    if len(uploaded) == 0:
        raise ValueError("No file was uploaded. Please upload one FASTA file.")
    if len(uploaded) > 1:
        raise ValueError(
            "Multiple files were uploaded. Please upload exactly one FASTA "
            "file per run."
        )

    file_name, file_bytes = next(iter(uploaded.items()))
    if not file_name.lower().endswith((".fasta", ".fa", ".fna")):
        raise ValueError(
            f"Uploaded file '{file_name}' is not a FASTA file. "
            "Please upload a file ending in .fasta, .fa, or .fna."
        )

    fasta_text = file_bytes.decode("utf-8")
    records = list(SeqIO.parse(io.StringIO(fasta_text), "fasta"))
    if not records:
        raise ValueError(
            "The uploaded FASTA file contains no sequences. Please check the "
            "file format and try again."
        )

    parsed = []
    for index, record in enumerate(records, start=1):
        seq_id = record.id.strip() if record.id else f"sequence_{index}"
        parsed.append(
            {
                "sequence_id": seq_id,
                "sequence": sanitize_sequence(str(record.seq)),
            }
        )
    return parsed


def validate_sequences(
    sequence_records: list[dict],
    max_sequences_allowed: int,
    max_sequence_length_allowed: int,
) -> None:
    """Validate sequence count, content, and length with friendly errors."""
    if max_sequences_allowed < 1:
        raise ValueError("MAX_SEQUENCES_ALLOWED must be at least 1.")
    if max_sequence_length_allowed < 1:
        raise ValueError("MAX_SEQUENCE_LENGTH_ALLOWED must be at least 1.")

    sequence_count = len(sequence_records)
    if sequence_count == 0:
        raise ValueError("No sequences were found after parsing the input.")

    if sequence_count > max_sequences_allowed:
        raise ValueError(
            f"You provided {sequence_count} sequences, but "
            f"MAX_SEQUENCES_ALLOWED is set to {max_sequences_allowed}. "
            "Please upload fewer sequences or increase the limit carefully."
        )

    validation_errors = []
    for record in sequence_records:
        seq_id = record["sequence_id"]
        sequence = record["sequence"]

        if len(sequence) == 0:
            validation_errors.append(
                f"Sequence '{seq_id}' is empty after cleaning."
            )
            continue

        if len(sequence) > max_sequence_length_allowed:
            validation_errors.append(
                f"Sequence '{seq_id}' has length {len(sequence)}, which exceeds "
                f"the limit ({max_sequence_length_allowed})."
            )

        invalid_characters = sorted(set(sequence) - VALID_AMINO_ACIDS)
        if invalid_characters:
            bad_chars = "".join(invalid_characters)
            validation_errors.append(
                f"Sequence '{seq_id}' contains invalid amino-acid characters: "
                f"'{bad_chars}'."
            )

    if validation_errors:
        error_lines = "\n".join(f"- {msg}" for msg in validation_errors)
        raise ValueError(
            "Input validation failed. Please fix the following issue(s):\n"
            f"{error_lines}"
        )


if INPUT_MODE == "Raw sequence":
    parsed_records = parse_sequences_from_raw(RAW_SEQUENCE)
else:
    parsed_records = parse_sequences_from_uploaded_fasta()

validate_sequences(
    sequence_records=parsed_records,
    max_sequences_allowed=int(MAX_SEQUENCES_ALLOWED),
    max_sequence_length_allowed=int(MAX_SEQUENCE_LENGTH_ALLOWED),
)

used_safe_ids: set[str] = set()
VALIDATED_RECORDS = []
for index, record in enumerate(parsed_records, start=1):
    safe_id = make_safe_id(record["sequence_id"], used_safe_ids, index)
    VALIDATED_RECORDS.append(
        {
            "sequence_id": record["sequence_id"],
            "safe_id": safe_id,
            "sequence": record["sequence"],
            "sequence_length": len(record["sequence"]),
        }
    )

SAFE_TO_ORIGINAL_ID = {
    record["safe_id"]: record["sequence_id"]
    for record in VALIDATED_RECORDS
}
SEQUENCE_LENGTH_BY_SAFE_ID = {
    record["safe_id"]: record["sequence_length"]
    for record in VALIDATED_RECORDS
}

input_preview_df = pd.DataFrame(
    {
        "sequence_id": [r["sequence_id"] for r in VALIDATED_RECORDS],
        "safe_id": [r["safe_id"] for r in VALIDATED_RECORDS],
        "sequence_length": [r["sequence_length"] for r in VALIDATED_RECORDS],
    }
)

display(input_preview_df)
print(f"Validated {len(VALIDATED_RECORDS)} sequence(s).")


In [ ]:
#@title 7) Generate ProtT5 embeddings (8 SimCSE-style variants)
if not VALIDATED_RECORDS:
    raise ValueError(
        "No validated sequences found. Please run the input validation cell first."
    )

if USE_VARIANTS:
    EMBEDDING_NUM_VARIANTS = 8
else:
    EMBEDDING_NUM_VARIANTS = 1
    print(
        "Model config indicates use_variants=False. Generating one deterministic "
        "embedding per sequence."
    )

CURRENT_RUN_ID = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
CURRENT_EMBEDDING_DIR = EMBEDDING_ROOT_DIR / f"run_{CURRENT_RUN_ID}"
CURRENT_EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

print(
    "Loading ProtT5 model and generating embeddings. "
    "This can take a few minutes on first run..."
)
print(
    f"Generating embeddings for {len(VALIDATED_RECORDS)} sequence(s) "
    f"with {EMBEDDING_NUM_VARIANTS} variant(s) each."
)
CURRENT_EMBEDDING_DIR, embedding_device = extract_prott5_embeddings(
    VALIDATED_RECORDS,
    CURRENT_EMBEDDING_DIR,
    pooling_type=POOLING_TYPE,
    num_variants=EMBEDDING_NUM_VARIANTS,
    device_str="auto",
)
PACKED_EMBEDDINGS, PACKED_SEQUENCE_IDS, PACKED_METADATA = open_packed_embedding_dataset(
    CURRENT_EMBEDDING_DIR
)

print(f"ProtT5 used device: {embedding_device}")
print(f"Saved embeddings to: {CURRENT_EMBEDDING_DIR}")
print(f"Packed embeddings shape: {PACKED_EMBEDDINGS.shape}")
print(f"Packed sequence count: {len(PACKED_SEQUENCE_IDS)}")
print(f"Packed embedding array: {CURRENT_EMBEDDING_DIR / 'embeddings.npy'}")
print(f"Packed metadata file: {CURRENT_EMBEDDING_DIR / 'metadata.json'}")


In [ ]:
#@title 8) Run fold-ensemble prototype inference
if "CURRENT_EMBEDDING_DIR" not in globals():
    raise RuntimeError(
        "Embeddings are not available yet. Please run the embedding generation "
        "cell first."
    )

if THRESHOLD_OVERRIDE.strip():
    try:
        SELECTED_THRESHOLD = float(THRESHOLD_OVERRIDE)
    except ValueError as exc:
        raise ValueError(
            "THRESHOLD_OVERRIDE must be a numeric value between 0 and 1 "
            "(or left blank for auto threshold)."
        ) from exc
else:
    SELECTED_THRESHOLD = float(AUTO_THRESHOLD)

if not 0.0 <= SELECTED_THRESHOLD <= 1.0:
    raise ValueError(
        f"Threshold must be between 0 and 1, but got {SELECTED_THRESHOLD}."
    )

INFERENCE_DEVICE = resolve_device(MODEL_CONFIG)
if INFERENCE_DEVICE.type.startswith("cuda") and not torch.cuda.is_available():
    print("CUDA requested by config but no GPU is available. Falling back to CPU.")
    INFERENCE_DEVICE = torch.device("cpu")

logger = logging.getLogger("colab_prototype_inference")
logger.setLevel(logging.INFO)
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())

INFERENCE_SEQUENCE_IDS_SAFE, INFERENCE_PROBABILITIES = run_inference(
    embedding_dir=CURRENT_EMBEDDING_DIR,
    model_dir=MODEL_DIR,
    config=MODEL_CONFIG,
    is_single_stage=IS_SINGLE_STAGE,
    device=INFERENCE_DEVICE,
    pooling_type=POOLING_TYPE,
    batch_size=32,
    logger=logger,
)
INFERENCE_BINARY = (
    INFERENCE_PROBABILITIES >= SELECTED_THRESHOLD
).astype(np.int32)

print("Inference completed.")
print(f"- Sequences scored: {len(INFERENCE_SEQUENCE_IDS_SAFE)}")
print(f"- Threshold used: {SELECTED_THRESHOLD:.6f}")


In [ ]:
#@title 9) Build results table and save CSV
required_objects = [
    "INFERENCE_SEQUENCE_IDS_SAFE",
    "INFERENCE_PROBABILITIES",
    "INFERENCE_BINARY",
    "SELECTED_THRESHOLD",
]
for obj_name in required_objects:
    if obj_name not in globals():
        raise RuntimeError(
            f"Missing '{obj_name}'. Please run the inference cell before building "
            "results."
        )

RESULTS_DF = pd.DataFrame(
    {
        "sequence_id": [
            SAFE_TO_ORIGINAL_ID.get(safe_id, safe_id)
            for safe_id in INFERENCE_SEQUENCE_IDS_SAFE
        ],
        "sequence_length": [
            SEQUENCE_LENGTH_BY_SAFE_ID.get(safe_id, np.nan)
            for safe_id in INFERENCE_SEQUENCE_IDS_SAFE
        ],
        "probability": INFERENCE_PROBABILITIES,
        "binary_prediction": INFERENCE_BINARY,
    }
)
RESULTS_DF["prediction_label"] = np.where(
    RESULTS_DF["binary_prediction"] == 1,
    "Effector",
    "Non-Effector",
)
RESULTS_DF["threshold"] = float(SELECTED_THRESHOLD)

RESULTS_DF = RESULTS_DF[
    [
        "sequence_id",
        "sequence_length",
        "probability",
        "binary_prediction",
        "prediction_label",
        "threshold",
    ]
].sort_values(by="probability", ascending=False).reset_index(drop=True)

OUTPUT_CSV_PATH = OUTPUT_DIR / f"fungus_inference_results_{CURRENT_RUN_ID}.csv"
RESULTS_DF.to_csv(OUTPUT_CSV_PATH, index=False)

display(RESULTS_DF)
print(f"Saved results CSV: {OUTPUT_CSV_PATH}")
print(
    "Prediction counts -> "
    f"Effector: {int((RESULTS_DF['binary_prediction'] == 1).sum())}, "
    f"Non-Effector: {int((RESULTS_DF['binary_prediction'] == 0).sum())}"
)


In [ ]:
#@title 10) Download results CSV
if "OUTPUT_CSV_PATH" not in globals():
    raise RuntimeError(
        "Results CSV path not found. Please run the results cell first."
    )

if colab_files is None:
    print(
        "google.colab is not available in this environment. "
        f"You can manually download the file from: {OUTPUT_CSV_PATH}"
    )
else:
    colab_files.download(str(OUTPUT_CSV_PATH))
    print(f"Download started for: {OUTPUT_CSV_PATH.name}")


## How to Interpret the Output

- `probability`: model-estimated probability that a sequence is an effector.
- `binary_prediction`: `1` means **Effector**, `0` means **Non-Effector**.
- `prediction_label`: readable label version of `binary_prediction`.
- `threshold`: decision cutoff used for the binary call.

This notebook reads the bundled default threshold from
`pretrained_models/fungus_model/metadata.yml`.
That default was obtained using **Youden's J statistics on pooled out-of-fold
(OOF) predictions**.

If you want a different cutoff, set `THRESHOLD_OVERRIDE` in the input form.
If you need stricter predictions (fewer positives), increase the threshold.
If you need higher sensitivity (more positives), decrease the threshold.
